# Feature selection (capstone track)

Mirrors a **dedicated feature_selection notebook** workflow (like the Gurgaon project): combine **filter** methods (correlation, redundancy) with **embedded** signals (Random Forest importances) and **mutual information** vs `log1p(price)`.

**Outputs (from `src/feature_selection.py`):**
- `models/selected_features.json` — ranked scores + final column list  
- `data/processed/texas_houston_features_selected.csv` — reduced training table  

Downstream: `train.py` auto-loads `selected_features.json` when present.


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_selection import (
    correlation_with_target,
    drop_high_correlation_pairs,
    mutual_information_scores,
    random_forest_importance,
    select_features,
    DEFAULT_CANDIDATES,
)

feat_path = PROJECT_ROOT / "data/processed/texas_houston_features.csv"
df = pd.read_csv(feat_path, low_memory=False)
y_log = pd.Series(np.log1p(df["price"].astype(float)), index=df.index)
candidates = [c for c in DEFAULT_CANDIDATES if c in df.columns]
print("Candidates:", len(candidates))


## 1. Correlation with target (log price)


In [ ]:
corr_tgt = correlation_with_target(df, y_log, candidates)
fig = px.bar(corr_tgt.reset_index(), x="index", y=0, title="|Correlation| with log1p(price) (numeric features)")
fig.update_layout(template="plotly_white", xaxis_title="feature", yaxis_title="correlation")
fig.show()
display(corr_tgt.to_frame("corr").round(4))


## 2. Mutual information (non-linear strength)


In [ ]:
mi = mutual_information_scores(df, y_log, candidates)
fig = px.bar(mi.reset_index(), x="index", y=0, title="Mutual information vs log1p(price)")
fig.update_layout(template="plotly_white", xaxis_title="feature", yaxis_title="MI")
fig.show()


## 3. Redundant pairs (multicollinearity) — drop weaker vs target


In [ ]:
kept, pairs = drop_high_correlation_pairs(df, candidates, corr_tgt, threshold=0.92)
print("Dropped from high pairwise corr:", len(pairs), "pairs")
pairs[:15]


## 4. Random Forest importances (mixed-type quick encoding)


In [ ]:
imp = random_forest_importance(df, y_log, candidates)
fig = px.bar(imp.reset_index(), x="index", y=0, title="RF feature importance")
fig.update_layout(template="plotly_white", xaxis_title="feature")
fig.show()


## 5. Run full selector + inspect JSON


In [ ]:
report = select_features(
    input_csv="data/processed/texas_houston_features.csv",
    output_json="models/selected_features.json",
    output_csv="data/processed/texas_houston_features_selected.csv",
)
print("Selected count:", len(report["selected_features"]))
print(report["selected_features"])
